# Detección Automática de Zonas Deforestadas en Colombia
### Imágenes Satelitales y Deep Learning como Herramienta para Política Pública

**Curso:** Deep Learning — Maestría en Inteligencia Analítica de Datos (MIAD)  
**Repositorio:** [Andres-Acuna/dl_deforestation](https://github.com/Andres-Acuna/dl_deforestation)  
**Versión:** 2.0 — Transfer learning E2E funcional (smp encoder en Fase 1 y Fase 2)

---

**Mejoras respecto a v1:**
- Fase 1 usa el encoder de `segmentation-models-pytorch` directamente, garantizando transferencia de pesos a Fase 2
- Transferencia real de pesos del encoder entre fases (verificada por nombre de capa)
- Fine-tuning con 30% de tiles y 10 épocas para mejor convergencia
- Inferencia sobre imagen de Meta (área de entrenamiento)
- Visualizaciones corregidas y ampliadas

| Sección | Descripción |
|---------|-------------|
| 0 | Configuración del entorno |
| 1 | Dataset Planet Amazon |
| 2 | Imágenes Sentinel-2 — GEE |
| 3 | Rasters del IDEAM — máscaras |
| 4 | Pipeline de datos |
| 5 | Arquitectura del modelo (smp E2E) |
| 6 | Entrenamiento Fase 1 — Planet Amazon |
| 7 | Fine-tuning Fase 2 — Colombia |
| 8 | Evaluación y resultados |
| 9 | Prototipo de aplicación |


---
## 0. Configuración del Entorno

In [ ]:
import os, sys, subprocess, getpass
from pathlib import Path

EN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content/drive')

if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE = Path('/content/drive/MyDrive/deforestation_project')
    REPO = Path('/content/dl_deforestation')
    if REPO.exists():
        r = subprocess.run(['git','pull'], cwd=REPO, capture_output=True, text=True)
        print(f'Repo actualizado: {r.stdout.strip()}')
    else:
        subprocess.run(['git','clone',
            'https://github.com/Andres-Acuna/dl_deforestation.git',
            str(REPO)], capture_output=True)
    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))
else:
    RUTAS_LOCALES = {
        'camil' : Path(r'C:\Users\camil\Documentos\MIAD\Deep Learning\Proyecto'),
        'andres': Path(r'C:\Users\andres\Documents\dl_deforestation'),
        # 'nuevo': Path(r'ruta\al\proyecto'),
    }
    usuario = getpass.getuser().lower()
    BASE = next((r for n, r in RUTAS_LOCALES.items()
                 if n in usuario or r.exists()), None)
    if BASE is None:
        raise EnvironmentError(
            f'Usuario "{usuario}" no reconocido. '
            'Agrega tu ruta en RUTAS_LOCALES.')

assert BASE.exists(), f'Carpeta base no existe: {BASE}'
print(f'Entorno : {"Colab" if EN_COLAB else "Local"}')
print(f'Usuario : {getpass.getuser()}')
print(f'Base    : {BASE}')


In [ ]:
import torch
import numpy as np

DATA_DIR      = BASE / 'data'
RAW_PLANET    = DATA_DIR / 'raw' / 'planet'
RAW_COLOMBIA  = DATA_DIR / 'raw' / 'colombia'
PROC_COLOMBIA = DATA_DIR / 'processed' / 'colombia'
MASKS_DIR     = DATA_DIR / 'masks'
CKPT_DIR      = BASE / 'checkpoints'
OUTPUTS_DIR   = BASE / 'outputs'

# Checkpoints V2 — no sobreescribir los de V1
BEST_CKPT_V2 = CKPT_DIR / 'planet_best_v2.pt'
COL_CKPT_V2  = CKPT_DIR / 'colombia_best_v2.pt'

for d in [PROC_COLOMBIA, MASKS_DIR, CKPT_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'img_size'        : 224,
    'batch_size'      : 16,
    'num_workers'     : 0 if os.name == 'nt' else 4,
    'tile_size'       : 224,
    'tile_stride'     : 112,
    'planet_epochs'   : 10,
    'planet_lr'       : 1e-3,
    'num_classes'     : 17,
    'colombia_epochs' : 10,
    'colombia_sample' : 0.30,   # 30% de tiles para convergencia rapida
    'colombia_lr'     : 5e-4,
    'seed'            : 42,
}

PLANET_TAGS = [
    'agriculture','artisinal_mine','bare_ground','blooming',
    'blow_down','clear','cloudy','conventional_mine','cultivation',
    'habitation','haze','partly_cloudy','primary','road',
    'selective_logging','slash_burn','water'
]
DEFORESTATION_TAGS = {
    'agriculture','slash_burn','habitation','cultivation',
    'bare_ground','artisinal_mine','selective_logging','blow_down'
}

torch.manual_seed(CONFIG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Rutas del proyecto:')
for nombre, ruta in [('Planet raw',RAW_PLANET),('Colombia raw',RAW_COLOMBIA),
                     ('Mascaras',MASKS_DIR),('Checkpoints',CKPT_DIR)]:
    print(f'  [{"OK" if ruta.exists() else "NO ENCONTRADA"}] {nombre:<16} {ruta}')
print(f'\nDispositivo : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


---
## 1. Dataset Planet Amazon

El dataset contiene ~40.000 imágenes JPG de 256×256 px del Amazonas brasileño, etiquetadas con 17 clases de cobertura. El 37.6% corresponde a imágenes con al menos una etiqueta de deforestación.

Sirve como fuente de preentrenamiento: el encoder aprende representaciones espectrales antes de transferirse a la tarea de segmentación colombiana.

In [ ]:
import zipfile, pandas as pd, matplotlib.pyplot as plt
from collections import Counter
from PIL import Image
import rasterio

def contar_imagenes(d):
    return len(list(Path(d).rglob('*.jpg'))) + len(list(Path(d).rglob('*.tif')))

n = contar_imagenes(RAW_PLANET)
if n > 0:
    print(f'Dataset ya descomprimido ({n:,} imagenes).')
else:
    zips = list(RAW_PLANET.glob('*.zip'))
    if not zips:
        print('Descarga el dataset:')
        print('kaggle datasets download -d nikitarom/planets-dataset -p data/raw/planet/')
    else:
        print(f'Descomprimiendo {zips[0].name}...')
        with zipfile.ZipFile(zips[0],'r') as z:
            z.extractall(RAW_PLANET)
        print(f'Listo: {contar_imagenes(RAW_PLANET):,} imagenes.')

# Cargar CSV
csv_path = next(RAW_PLANET.rglob('train_classes.csv'), None) or \
           next(RAW_PLANET.rglob('train_v2.csv'), None)
assert csv_path, 'No se encontro el CSV de etiquetas.'

df = pd.read_csv(csv_path)
df['tags_list'] = df['tags'].str.split()
df['es_deforestada'] = df['tags_list'].apply(
    lambda t: int(bool(set(t) & DEFORESTATION_TAGS)))

TIF_DIR = next(RAW_PLANET.rglob('train-jpg'), None)
assert TIF_DIR, 'No se encontro la carpeta train-jpg.'

print(f'Total imagenes    : {len(df):,}')
print(f'Con deforestacion : {df["es_deforestada"].sum():,} ({df["es_deforestada"].mean():.1%})')


In [ ]:
# Distribucion de etiquetas
all_tags = [t for tags in df['tags_list'] for t in tags]
conteo   = Counter(all_tags)
tags_ord = sorted(conteo.items(), key=lambda x: -x[1])

fig, ax = plt.subplots(figsize=(10,5))
nombres = [t[0] for t in tags_ord]
valores = [t[1] for t in tags_ord]
colores = ['#c0392b' if n in DEFORESTATION_TAGS else '#2e7d32' for n in nombres]
ax.barh(nombres, valores, color=colores)
ax.set_xlabel('Frecuencia')
ax.set_title('Distribución de etiquetas — Planet Amazon')
ax.invert_yaxis()
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#c0392b', label='Relacionada con deforestación'),
    Patch(color='#2e7d32', label='Otras clases')
], loc='lower right')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'distribucion_etiquetas.png', dpi=120)
plt.show()


In [ ]:
# Visualizacion de muestras por clase
def leer_rgb(ruta):
    img = np.array(Image.open(ruta)).astype(np.float32)[:,:,:3] / 255.0
    return np.clip(img, 0, 1)

clases_muestra = ['primary','agriculture','slash_burn',
                  'habitation','bare_ground','water']
muestras = []
for clase in clases_muestra:
    filas = df[df['tags_list'].apply(lambda t: clase in t)]
    if len(filas) > 0:
        ruta = TIF_DIR / f'{filas.iloc[0]["image_name"]}.jpg'
        if ruta.exists():
            muestras.append((clase, ruta))

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
fig.suptitle('Muestras del dataset Planet Amazon — color verdadero (RGB)',
             fontsize=13, fontweight='bold')
for ax, (clase, ruta) in zip(axes.flat, muestras):
    ax.imshow(leer_rgb(ruta))
    color = '#c0392b' if clase in DEFORESTATION_TAGS else '#2e7d32'
    ax.set_title(clase, fontsize=11, color=color, fontweight='bold')
    ax.axis('off')
for ax in axes.flat[len(muestras):]:
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'muestras_planet.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado en outputs/muestras_planet.png')


---
## 2. Imágenes Sentinel-2 para Colombia (Google Earth Engine)

Las imágenes ya fueron exportadas desde GEE y descargadas manualmente. Esta sección verifica su disponibilidad. Las exportaciones cubren el departamento del **Meta** (2022), que es la fuente de datos principal para el fine-tuning al contar con máscaras de deforestación validadas.

> Para re-exportar: https://code.earthengine.google.com/tasks

In [ ]:
tifs_colombia = list(RAW_COLOMBIA.rglob('*.tif'))

if len(tifs_colombia) > 0:
    meta  = [f for f in tifs_colombia if 'meta' in f.name]
    otros = [f for f in tifs_colombia if 'meta' not in f.name]
    print(f'Imagenes Meta     : {len(meta)} tiles')
    print(f'Otras imagenes    : {len(otros)} tiles')
    print(f'Total disponible  : {len(tifs_colombia)} archivos')
    total_gb = sum(f.stat().st_size for f in tifs_colombia) / 1e9
    print(f'Espacio total     : {total_gb:.1f} GB')
else:
    print(f'Sin imagenes en {RAW_COLOMBIA}')
    print('Descarga los GeoTIFF desde Drive y muevelos a esa carpeta.')


---
## 3. Máscaras de Deforestación (Hansen/IDEAM)

El raster Hansen `lossyear` indica el año de pérdida boscosa por píxel. Se reproyecta al CRS y resolución de cada tile Sentinel-2 para generar máscaras binarias (0=bosque, 1=deforestado) usadas como etiquetas de entrenamiento.

**Cobertura de deforestación detectada en Meta (2022):** ~209.000 ha

In [ ]:
from rasterio.warp import reproject, Resampling

def alinear_mascara(ruta_mascara, ruta_imagen_ref, ruta_salida):
    with rasterio.open(ruta_imagen_ref) as ref:
        crs_dst = ref.crs
        tf_dst  = ref.transform
        W, H    = ref.width, ref.height
    with rasterio.open(ruta_mascara) as src:
        datos = np.zeros((1, H, W), dtype=np.uint8)
        reproject(source=rasterio.band(src,1), destination=datos,
                  src_transform=src.transform, src_crs=src.crs,
                  dst_transform=tf_dst, dst_crs=crs_dst,
                  resampling=Resampling.nearest)
    datos_bin = (datos > 0).astype(np.uint8)
    with rasterio.open(ruta_salida, 'w', driver='GTiff',
                       height=H, width=W, count=1, dtype=np.uint8,
                       crs=crs_dst, transform=tf_dst) as dst:
        dst.write(datos_bin)
    return int(datos_bin.sum())


mask_raw = MASKS_DIR / 'ideam_meta_2022.tif'
imagenes_meta = sorted(RAW_COLOMBIA.glob('s2_meta_*.tif'))
pares_validos = []

print(f'Tiles Meta disponibles: {len(imagenes_meta)}')
for img_path in imagenes_meta:
    salida = MASKS_DIR / f'mask_{img_path.stem}.tif'
    if salida.exists():
        with rasterio.open(salida) as s:
            n_def = int((s.read(1) > 0).sum())
        print(f'  Ya existe: {salida.name}  ({n_def:,} px)')
    else:
        n_def = alinear_mascara(mask_raw, img_path, salida)
        print(f'  Creada: {salida.name}  ({n_def:,} px)')
    if n_def > 0:
        pares_validos.append((img_path, salida))

print(f'\nPares validos para entrenamiento: {len(pares_validos)}')


---
## 4. Pipeline de Datos

Generación de tiles de 224×224 px con stride del 50% sobre las imágenes Sentinel-2 de Meta. Se descartan tiles con más del 20% de píxeles inválidos.

In [ ]:
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch.nn as nn

TAG_TO_IDX = {tag: i for i, tag in enumerate(PLANET_TAGS)}


def get_transforms_planet(modo='train'):
    comunes = [A.Normalize(mean=[0.485,0.456,0.406],
                            std=[0.229,0.224,0.225]), ToTensorV2()]
    if modo == 'train':
        return A.Compose([
            A.RandomCrop(CONFIG['img_size'], CONFIG['img_size']),
            A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.OneOf([A.GaussNoise(p=1), A.GaussianBlur(p=1)], p=0.2),
            A.ColorJitter(brightness=0.2, contrast=0.2, p=0.3),
        ] + comunes)
    return A.Compose([A.CenterCrop(CONFIG['img_size'], CONFIG['img_size'])] + comunes)


class PlanetDataset(Dataset):
    def __init__(self, df, jpg_dir, modo='train'):
        n = len(df)
        cuts = {'train':(0,int(n*.8)),'val':(int(n*.8),int(n*.9)),'test':(int(n*.9),n)}
        a, b = cuts[modo]
        self.df = df.iloc[a:b].reset_index(drop=True)
        self.jpg_dir   = Path(jpg_dir)
        self.transform = get_transforms_planet(modo)

    def _enc(self, tags):
        v = np.zeros(len(PLANET_TAGS), dtype=np.float32)
        for t in tags:
            if t in TAG_TO_IDX: v[TAG_TO_IDX[t]] = 1.0
        return v

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        import torch
        fila = self.df.iloc[idx]
        img  = np.array(Image.open(
            self.jpg_dir / f'{fila["image_name"]}.jpg'
        )).astype(np.float32)[:,:,:3] / 255.0
        aug = self.transform(image=img)
        return aug['image'], torch.tensor(self._enc(fila['tags_list']))


PIN = torch.cuda.is_available()
ds_train = PlanetDataset(df, TIF_DIR, 'train')
ds_val   = PlanetDataset(df, TIF_DIR, 'val')
ds_test  = PlanetDataset(df, TIF_DIR, 'test')
dl_train = DataLoader(ds_train, CONFIG['batch_size'], shuffle=True,
                      num_workers=0, pin_memory=PIN)
dl_val   = DataLoader(ds_val,   CONFIG['batch_size'], shuffle=False,
                      num_workers=0, pin_memory=PIN)
dl_test  = DataLoader(ds_test,  CONFIG['batch_size'], shuffle=False,
                      num_workers=0, pin_memory=PIN)
print(f'Train : {len(ds_train):,}  ({len(dl_train)} batches)')
print(f'Val   : {len(ds_val):,}  ({len(dl_val)} batches)')
imgs, labels = next(iter(dl_train))
print(f'Batch: imgs={imgs.shape}  labels={labels.shape}')


In [ ]:
# Generacion de tiles Colombia
import shutil

def generar_tiles(ruta_imagen, ruta_mascara, dir_salida,
                  tam_tile=224, stride=112, min_validos=0.8):
    dir_salida = Path(dir_salida)
    (dir_salida/'images').mkdir(parents=True, exist_ok=True)
    (dir_salida/'masks').mkdir(parents=True, exist_ok=True)
    existentes = len(list((dir_salida/'images').glob('*.npy')))
    with rasterio.open(ruta_imagen) as img_src, \
         rasterio.open(ruta_mascara) as mask_src:
        H, W = img_src.height, img_src.width
        n_ok = n_skip = 0
        for fila in range(0, H-tam_tile, stride):
            for col in range(0, W-tam_tile, stride):
                w  = rasterio.windows.Window(col, fila, tam_tile, tam_tile)
                ti = img_src.read(window=w).astype(np.float32)
                tm = mask_src.read(1, window=w).astype(np.uint8)
                if not np.isfinite(ti).all() or \
                   np.isfinite(ti).all(axis=0).mean() < min_validos:
                    n_skip += 1; continue
                idx = existentes + n_ok
                np.save(dir_salida/'images'/f'tile_{idx:05d}.npy', ti)
                np.save(dir_salida/'masks' /f'tile_{idx:05d}.npy', tm)
                n_ok += 1
    print(f'  {ruta_imagen.name}: {n_ok:,} tiles | {n_skip:,} descartados')
    return n_ok


tiles_exist = list((PROC_COLOMBIA/'images').rglob('*.npy'))
if len(tiles_exist) > 0:
    print(f'Tiles ya generados: {len(tiles_exist):,}. Se omite.')
elif not pares_validos:
    print('Sin pares validos. Verifica Seccion 3.')
else:
    total = 0
    for img_p, mask_p in pares_validos:
        print(f'Procesando: {img_p.name}')
        total += generar_tiles(img_p, mask_p, PROC_COLOMBIA)
    print(f'\nTotal tiles: {total:,}')
    tiles_con_def = sum(1 for p in
        list((PROC_COLOMBIA/'masks').glob('*.npy'))[:200]
        if np.load(p).sum() > 0)
    print(f'Con deforestacion (muestra 200): {tiles_con_def}/200')


In [ ]:
# Muestra acotada para fine-tuning rapido
import random

todas = sorted((PROC_COLOMBIA/'images').glob('*.npy'))
random.seed(CONFIG['seed'])
muestra_col = random.sample(todas, int(len(todas)*CONFIG['colombia_sample']))

print(f'Total tiles    : {len(todas):,}')
print(f'Muestra ({CONFIG["colombia_sample"]:.0%}) : {len(muestra_col):,}')
print(f'Batches train  : {int(len(muestra_col)*0.70)//16} por epoca')
print(f'Tiempo aprox   : ~{int(len(muestra_col)*0.70)//16*7//60} min total '
      f'({CONFIG["colombia_epochs"]} epocas)')


class ColombiaDataset(Dataset):
    def __init__(self, rutas, modo='train'):
        n = len(rutas)
        cuts = {'train':(0,int(n*.70)),'val':(int(n*.70),int(n*.85)),'test':(int(n*.85),n)}
        a, b = cuts[modo]
        self.rutas = rutas[a:b]
        comunes = [A.Normalize(mean=[0.485,0.456,0.406,0.5],
                               std=[0.229,0.224,0.225,0.25]), ToTensorV2()]
        self.transform = A.Compose(
            ([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
              A.RandomRotate90(p=0.5)] if modo=='train' else []) + comunes)

    def __len__(self): return len(self.rutas)

    def __getitem__(self, idx):
        import torch
        ruta_img  = self.rutas[idx]
        ruta_mask = PROC_COLOMBIA/'masks'/ruta_img.name
        img  = np.clip(np.load(ruta_img).astype(np.float32)/10000.0, 0, 1)
        mask = np.load(ruta_mask)
        aug  = self.transform(image=np.transpose(img,(1,2,0)),
                              mask=mask.astype(np.float32))
        return aug['image'], aug['mask'].long()


---
## 5. Arquitectura del Modelo — Transfer Learning E2E

**Cambio clave respecto a v1:** ambas fases usan el encoder de `segmentation-models-pytorch`, garantizando que los nombres de las capas sean idénticos y la transferencia de pesos entre fases sea directa.

- **Fase 1:** encoder EfficientNet-B4 de smp + head de clasificación global → Planet Amazon
- **Fase 2:** mismo encoder (pesos transferidos) + decoder U-Net → segmentación Colombia

Los pesos del encoder se transfieren capa a capa por nombre (`encoder.*`).

In [ ]:
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_encoder


class ModeloPlanetSMP(nn.Module):
    """
    Clasificador multi-label usando el encoder de smp.
    Al compartir la misma convencion de nombres que U-Net,
    los pesos son directamente transferibles en Fase 2.
    """
    def __init__(self, num_classes=17, dropout=0.3):
        super().__init__()
        self.encoder = get_encoder(
            'efficientnet-b4',
            in_channels=3,
            depth=5,
            weights='imagenet'
        )
        feat_dim = self.encoder.out_channels[-1]  # 448 para EffNet-B4
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout/2),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.encoder(x)
        x = self.pool(features[-1])
        x = torch.flatten(x, 1)
        return self.head(x)


modelo_planet = ModeloPlanetSMP(CONFIG['num_classes']).to(DEVICE)

with torch.no_grad():
    x_test = torch.randn(2, 3, CONFIG['img_size'], CONFIG['img_size']).to(DEVICE)
    out    = modelo_planet(x_test)

n_params = sum(p.numel() for p in modelo_planet.parameters() if p.requires_grad)
print(f'Entrada    : {x_test.shape}')
print(f'Salida     : {out.shape}  (esperado: [2, {CONFIG["num_classes"]}])')
print(f'Parametros : {n_params:,}')
print(f'Feat dim   : {modelo_planet.encoder.out_channels[-1]}')


---
## 6. Entrenamiento — Fase 1: Planet Amazon

Preentrenamiento del encoder sobre Planet Amazon. Métricas de referencia de la v1: **F2=0.9206** (época 10), confirmando que el encoder aprende representaciones discriminativas de cobertura forestal.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import fbeta_score


def entrenar_planet(modelo, dl_train, dl_val, config, ruta_ckpt):
    pos_w    = torch.ones(config['num_classes']) * 2.0
    criterio = nn.BCEWithLogitsLoss(pos_weight=pos_w.to(DEVICE))
    opt   = AdamW(modelo.parameters(), lr=config['planet_lr'], weight_decay=1e-4)
    sched = OneCycleLR(opt, max_lr=config['planet_lr'],
                       epochs=config['planet_epochs'],
                       steps_per_epoch=len(dl_train))
    mejor_f2  = 0.0
    historial = {'loss': [], 'f2': []}

    for epoca in range(config['planet_epochs']):
        modelo.train()
        loss_total = 0.0
        for imgs, labels in dl_train:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            loss = criterio(modelo(imgs), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            opt.step(); sched.step()
            loss_total += loss.item()

        modelo.eval()
        preds_all, labels_all = [], []
        with torch.no_grad():
            for imgs, labels in dl_val:
                preds_all.append(torch.sigmoid(modelo(imgs.to(DEVICE))).cpu().numpy())
                labels_all.append(labels.numpy())
        preds  = (np.vstack(preds_all) > 0.2).astype(int)
        labels_np = np.vstack(labels_all)
        f2 = fbeta_score(labels_np, preds, beta=2, average='samples', zero_division=0)
        lm = loss_total / len(dl_train)
        historial['loss'].append(lm)
        historial['f2'].append(f2)
        print(f'Epoca {epoca+1:>3}/{config["planet_epochs"]}  loss={lm:.4f}  F2={f2:.4f}')
        if f2 > mejor_f2:
            mejor_f2 = f2
            torch.save({'epoch':epoca,'model_state_dict':modelo.state_dict(),'f2':mejor_f2}, ruta_ckpt)
            print(f'  Checkpoint guardado (F2={mejor_f2:.4f})')
    return historial


In [ ]:
if BEST_CKPT_V2.exists():
    ckpt = torch.load(BEST_CKPT_V2, map_location=DEVICE)
    modelo_planet.load_state_dict(ckpt['model_state_dict'])
    print(f'Checkpoint V2 cargado — F2={ckpt["f2"]:.4f}  epoca={ckpt["epoch"]+1}')
    historial_planet = None
else:
    print(f'Iniciando Fase 1 en {DEVICE}...')
    historial_planet = entrenar_planet(
        modelo_planet, dl_train, dl_val, CONFIG, BEST_CKPT_V2)


In [ ]:
# Curvas de aprendizaje Fase 1
if historial_planet is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    epocas = range(1, len(historial_planet['loss'])+1)
    ax1.plot(epocas, historial_planet['loss'], color='#c0392b', marker='o', ms=4)
    ax1.set_title('Pérdida de entrenamiento (Fase 1)')
    ax1.set_xlabel('Época'); ax1.set_ylabel('BCE Loss'); ax1.grid(alpha=0.3)
    ax2.plot(epocas, historial_planet['f2'], color='#2e7d32', marker='o', ms=4)
    ax2.set_title('F2-score en validación (Fase 1)')
    ax2.set_xlabel('Época'); ax2.set_ylabel('F2'); ax2.grid(alpha=0.3)
    ax2.axhline(y=0.9, color='gray', linestyle='--', alpha=0.5, label='F2=0.9')
    ax2.legend()
    plt.suptitle('Entrenamiento Fase 1 — Planet Amazon', fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR/'curvas_fase1_v2.png', dpi=120)
    plt.show()


---
## 7. Fine-tuning — Fase 2: Colombia (Transfer Learning E2E)

Los pesos del encoder de Fase 1 se transfieren directamente al encoder de la U-Net. Al usar la misma librería (smp) en ambas fases, la correspondencia de nombres es exacta y la transferencia es completa.

Se aplican learning rates diferenciales: el encoder recibe 10× menos que el decoder para preservar las representaciones aprendidas.

In [ ]:
def construir_unet_colombia(ruta_ckpt_planet=None):
    modelo = smp.Unet(
        encoder_name='efficientnet-b4',
        encoder_weights='imagenet',
        in_channels=4,
        classes=1,
        activation=None
    )

    if ruta_ckpt_planet and Path(ruta_ckpt_planet).exists():
        ckpt          = torch.load(ruta_ckpt_planet, map_location='cpu')
        estado_unet   = modelo.state_dict()
        estado_planet = ckpt['model_state_dict']

        cargados = omitidos = 0
        for k, v in estado_planet.items():
            # Ambos modelos usan smp: los nombres encoder.* coinciden directamente
            if k.startswith('encoder.') and k in estado_unet:
                if estado_unet[k].shape == v.shape:
                    estado_unet[k] = v
                    cargados += 1
                else:
                    omitidos += 1

        modelo.load_state_dict(estado_unet)
        print(f'Pesos transferidos   : {cargados} capas del encoder')
        print(f'Omitidas (shape)     : {omitidos}')
        if cargados == 0:
            print('ADVERTENCIA: 0 capas transferidas. '
                  'Verifica que el checkpoint es de ModeloPlanetSMP.')
    else:
        print('Sin checkpoint — usando pesos ImageNet.')

    return modelo


def entrenar_colombia(modelo, dl_train, dl_val, config, ruta_ckpt):
    dice_loss = smp.losses.DiceLoss(mode='binary')
    bce_loss  = smp.losses.SoftBCEWithLogitsLoss()

    def criterio(pred, target):
        return dice_loss(pred, target) + bce_loss(pred, target)

    opt = AdamW([
        {'params': modelo.encoder.parameters(),        'lr': config['colombia_lr']*0.1},
        {'params': modelo.decoder.parameters(),        'lr': config['colombia_lr']},
        {'params': modelo.segmentation_head.parameters(),'lr': config['colombia_lr']},
    ], weight_decay=1e-4)

    mejor_iou = 0.0
    historial  = {'loss': [], 'iou': []}

    for epoca in range(config['colombia_epochs']):
        modelo.train()
        loss_total = 0.0
        for imgs, masks in dl_train:
            imgs  = imgs.to(DEVICE)
            masks = masks.float().unsqueeze(1).to(DEVICE)
            opt.zero_grad()
            loss = criterio(modelo(imgs), masks)
            loss.backward()
            opt.step()
            loss_total += loss.item()

        modelo.eval()
        ious = []
        with torch.no_grad():
            for imgs, masks in dl_val:
                preds = torch.sigmoid(modelo(imgs.to(DEVICE))) > 0.5
                masks = masks.unsqueeze(1).bool()
                inter = (preds.cpu() & masks).float().sum()
                union = (preds.cpu() | masks).float().sum()
                ious.append((inter/(union+1e-8)).item())

        iou_m  = float(np.mean(ious))
        loss_m = loss_total / len(dl_train)
        historial['loss'].append(loss_m)
        historial['iou'].append(iou_m)
        print(f'Epoca {epoca+1:>3}/{config["colombia_epochs"]}  '
              f'loss={loss_m:.4f}  IoU={iou_m:.4f}')

        if iou_m > mejor_iou:
            mejor_iou = iou_m
            torch.save({'epoch':epoca,'model_state_dict':modelo.state_dict(),
                        'iou':mejor_iou}, ruta_ckpt)
            print(f'  Checkpoint guardado (IoU={mejor_iou:.4f})')

    return historial


In [ ]:
if COL_CKPT_V2.exists():
    modelo_colombia = construir_unet_colombia(BEST_CKPT_V2).to(DEVICE)
    ckpt = torch.load(COL_CKPT_V2, map_location=DEVICE)
    modelo_colombia.load_state_dict(ckpt['model_state_dict'])
    print(f'Checkpoint Colombia V2 cargado — IoU={ckpt["iou"]:.4f}  epoca={ckpt["epoch"]+1}')
    historial_colombia = None

else:
    ds_col_train = ColombiaDataset(muestra_col, 'train')
    ds_col_val   = ColombiaDataset(muestra_col, 'val')
    dl_col_train = DataLoader(ds_col_train, 16, shuffle=True,  num_workers=0, pin_memory=True)
    dl_col_val   = DataLoader(ds_col_val,   16, shuffle=False, num_workers=0, pin_memory=True)

    print(f'Train : {len(ds_col_train):,}  ({len(dl_col_train)} batches por epoca)')
    print(f'Val   : {len(ds_col_val):,}')

    modelo_colombia = construir_unet_colombia(BEST_CKPT_V2).to(DEVICE)
    print(f'Iniciando fine-tuning en {DEVICE}...')
    historial_colombia = entrenar_colombia(
        modelo_colombia, dl_col_train, dl_col_val, CONFIG, COL_CKPT_V2)


In [ ]:
# Curvas de aprendizaje Fase 2
if historial_colombia is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    epocas = range(1, len(historial_colombia['loss'])+1)
    ax1.plot(epocas, historial_colombia['loss'], color='#c0392b', marker='o', ms=4)
    ax1.set_title('Pérdida de entrenamiento (Fase 2)')
    ax1.set_xlabel('Época'); ax1.set_ylabel('Dice + BCE Loss'); ax1.grid(alpha=0.3)
    ax2.plot(epocas, historial_colombia['iou'], color='#1565c0', marker='o', ms=4)
    ax2.set_title('IoU en validación (Fase 2 — Colombia)')
    ax2.set_xlabel('Época'); ax2.set_ylabel('IoU'); ax2.grid(alpha=0.3)
    plt.suptitle('Fine-tuning Fase 2 — Meta, Colombia', fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR/'curvas_fase2_v2.png', dpi=120)
    plt.show()


---
## 8. Evaluación y Resultados

Evaluación del modelo sobre el conjunto de prueba con reporte de precisión, recall, F1-score e IoU por clase. Se incluye visualización de predicciones sobre tiles reales de Colombia.

In [ ]:
from sklearn.metrics import classification_report

def evaluar(modelo, dataloader, umbral=0.5):
    modelo.eval()
    preds_all, masks_all = [], []
    with torch.no_grad():
        for imgs, masks in dataloader:
            logits = modelo(imgs.to(DEVICE))
            preds  = (torch.sigmoid(logits) > umbral).cpu().numpy()
            preds_all.append(preds.flatten())
            masks_all.append(masks.numpy().flatten())
    y_pred = np.concatenate(preds_all)
    y_true = np.concatenate(masks_all)
    print(classification_report(y_true, y_pred,
          target_names=['Bosque','Deforestado'], zero_division=0))
    for clase, nombre in enumerate(['Bosque','Deforestado']):
        inter = ((y_pred==clase) & (y_true==clase)).sum()
        union = ((y_pred==clase) | (y_true==clase)).sum()
        print(f'IoU {nombre:<15}: {inter/(union+1e-8):.4f}')
    return y_true, y_pred


if 'modelo_colombia' in dir():
    ds_col_test = ColombiaDataset(muestra_col, 'test')
    dl_col_test = DataLoader(ds_col_test, 16, shuffle=False, num_workers=0)
    print(f'Test : {len(ds_col_test):,} tiles')
    y_true, y_pred = evaluar(modelo_colombia, dl_col_test)
else:
    print('Pendiente: completar Seccion 7.')


In [ ]:
# Visualizacion de predicciones — comparacion imagen / mascara real / prediccion
def visualizar_predicciones(modelo, dataset, n=6, umbral=0.5):
    modelo.eval()
    # Seleccionar tiles que tengan deforestacion real para mejor visualizacion
    indices_con_def = [
        i for i in range(len(dataset))
        if np.load(PROC_COLOMBIA/'masks'/dataset.rutas[i].name).sum() > 100
    ]
    if len(indices_con_def) < n:
        indices_con_def = list(range(min(n, len(dataset))))
    indices = indices_con_def[:n]

    fig, axes = plt.subplots(n, 3, figsize=(13, n*3.5))
    fig.suptitle('Predicciones del modelo — Meta, Colombia\n'
                 '(tiles con deforestacion real)', fontsize=13, fontweight='bold')

    mean = np.array([0.485,0.456,0.406])
    std  = np.array([0.229,0.224,0.225])

    for fila, idx in enumerate(indices):
        img_t, mask_t = dataset[idx]
        with torch.no_grad():
            prob = torch.sigmoid(
                modelo(img_t.unsqueeze(0).to(DEVICE))
            ).cpu().squeeze().numpy()
            pred = (prob > umbral).astype(np.uint8)

        rgb = np.clip(img_t.numpy().transpose(1,2,0)[:,:,:3]*std+mean, 0, 1)
        mask_np = mask_t.numpy()

        # IoU del tile
        inter = (pred & mask_np).sum()
        union = (pred | mask_np).sum()
        tile_iou = inter/(union+1e-8)

        for col, (dato, titulo, cmap, vmin, vmax) in enumerate([
            (rgb,     'Imagen Sentinel-2 (RGB)',   None,        None, None),
            (mask_np, 'Máscara real (IDEAM)',       'RdYlGn_r',  0,    1),
            (pred,    f'Predicción (IoU={tile_iou:.2f})', 'RdYlGn_r', 0, 1),
        ]):
            axes[fila,col].imshow(dato, cmap=cmap, vmin=vmin, vmax=vmax)
            if fila == 0:
                axes[fila,col].set_title(titulo, fontsize=11, fontweight='bold')
            axes[fila,col].axis('off')

    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR/'predicciones_v2.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Guardado en outputs/predicciones_v2.png')


if 'modelo_colombia' in dir():
    visualizar_predicciones(modelo_colombia, ds_col_test, n=6)
else:
    print('Pendiente: completar Seccion 7.')


---
## 9. Prototipo de Aplicación

Inferencia sobre una imagen Sentinel-2 completa de **Meta** — la misma área usada para el entrenamiento. El procesamiento por parches con promedio en zonas de solapamiento garantiza predicciones coherentes en los bordes. El output es un GeoTIFF compatible con QGIS.

In [ ]:
def inferencia_completa(ruta_imagen, modelo, tam_tile=224, stride=112, umbral=0.5):
    transform = A.Compose([
        A.Normalize(mean=[0.485,0.456,0.406,0.5],
                    std=[0.229,0.224,0.225,0.25]),
        ToTensorV2()
    ])
    modelo.eval()
    with rasterio.open(ruta_imagen) as src:
        imagen = src.read([1,2,3,4]).astype(np.float32)
        perfil = src.profile
        res_m  = abs(src.transform.a)
    _, H, W = imagen.shape
    mapa_prob   = np.zeros((H,W), dtype=np.float32)
    mapa_conteo = np.zeros((H,W), dtype=np.float32)
    total_tiles = ((H-tam_tile)//stride) * ((W-tam_tile)//stride)
    procesados  = 0
    for fila in range(0, H-tam_tile+1, stride):
        for col in range(0, W-tam_tile+1, stride):
            tile = np.clip(
                imagen[:,fila:fila+tam_tile,col:col+tam_tile]/10000.0, 0, 1)
            if not np.isfinite(tile).all(): continue
            aug = transform(image=np.transpose(tile,(1,2,0)))
            with torch.no_grad():
                prob = torch.sigmoid(
                    modelo(aug['image'].unsqueeze(0).to(DEVICE))
                ).cpu().squeeze().numpy()
            mapa_prob[fila:fila+tam_tile,col:col+tam_tile]   += prob
            mapa_conteo[fila:fila+tam_tile,col:col+tam_tile] += 1.0
            procesados += 1
            if procesados % 500 == 0:
                print(f'  {procesados}/{total_tiles} tiles ({procesados/total_tiles*100:.0f}%)')
    mapa_prob  /= np.maximum(mapa_conteo, 1)
    mapa_bin    = (mapa_prob > umbral).astype(np.uint8)
    area_ha     = mapa_bin.sum() * (res_m**2) / 10_000
    return mapa_prob, mapa_bin, area_ha, perfil


if 'modelo_colombia' not in dir():
    print('Pendiente: completar Seccion 7.')
else:
    # Usar imagen de Meta — misma area de entrenamiento
    candidatos_meta = sorted(RAW_COLOMBIA.rglob('s2_meta_2022-0000000000-0000011776.tif'))
    if not candidatos_meta:
        candidatos_meta = sorted(RAW_COLOMBIA.rglob('s2_meta*.tif'))
    assert candidatos_meta, 'No se encontraron imagenes de Meta.'
    img_meta = candidatos_meta[0]

    print(f'Procesando: {img_meta.name}')
    print('(Puede tomar 3-10 min segun resolucion de la imagen)')
    mapa_prob, mapa_bin, area_ha, perfil = inferencia_completa(img_meta, modelo_colombia)
    print(f'\nArea deforestada estimada: {area_ha:,.0f} hectareas')

    # Guardar GeoTIFF
    ruta_geotiff = OUTPUTS_DIR / f'prediccion_meta_v2.tif'
    perfil.update(count=1, dtype=rasterio.uint8)
    with rasterio.open(ruta_geotiff, 'w', **perfil) as dst:
        dst.write(mapa_bin[np.newaxis,:,:])
    print(f'GeoTIFF guardado: {ruta_geotiff}')
    print('Compatible con QGIS para analisis espacial.')


In [ ]:
# Visualizacion final — mapa de prediccion
if 'mapa_prob' in dir():
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f'Mapa de Deforestación — Meta, Colombia (2022)\n'
                 f'Área estimada: {area_ha:,.0f} ha', fontsize=13, fontweight='bold')

    # Panel 1: probabilidad continua
    im1 = axes[0].imshow(mapa_prob, cmap='RdYlGn_r', vmin=0, vmax=1)
    axes[0].set_title('Probabilidad de deforestación', fontsize=11)
    plt.colorbar(im1, ax=axes[0], fraction=0.046)

    # Panel 2: mapa binario
    axes[1].imshow(mapa_bin, cmap='RdYlGn_r', vmin=0, vmax=1)
    axes[1].set_title('Mapa binario (umbral=0.5)', fontsize=11)

    # Panel 3: histograma de probabilidades
    probs_flat = mapa_prob[mapa_prob > 0].flatten()
    axes[2].hist(probs_flat, bins=50, color='#2e7d32', alpha=0.7, edgecolor='white')
    axes[2].axvline(x=0.5, color='#c0392b', linestyle='--', label='Umbral 0.5')
    axes[2].set_title('Distribución de probabilidades', fontsize=11)
    axes[2].set_xlabel('Probabilidad de deforestación')
    axes[2].set_ylabel('Frecuencia')
    axes[2].legend()

    for ax in axes[:2]: ax.axis('off')
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR/'mapa_deforestacion_v2.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Guardado en outputs/mapa_deforestacion_v2.png')
else:
    print('Pendiente: completar Seccion 9 (inferencia).')


---
## Notas de uso

### Diferencias V1 → V2
| Aspecto | V1 | V2 |
|---------|----|----|  
| Encoder Fase 1 | torchvision custom | smp encoder |
| Transferencia | 0 capas (incompatible) | 100% capas encoder |
| Sample Colombia | 20% | 30% |
| Épocas Fase 2 | 5 | 10 |
| Inferencia | Caquetá (incorrecto) | Meta (área entrenamiento) |
| Visualización | Básica | Tiles con deforestación + histograma |

### Guardar cambios al repositorio
```python
import subprocess
repo = '/content/dl_deforestation' if EN_COLAB else str(BASE)
subprocess.run(['git', 'config', 'user.email', 'correo@ejemplo.com'], cwd=repo)
subprocess.run(['git', 'config', 'user.name', 'Nombre'], cwd=repo)
subprocess.run(['git', 'add', 'notebooks/proyecto_deforestacion_v2.ipynb'], cwd=repo)
subprocess.run(['git', 'commit', '-m', 'feat: notebook v2 transfer learning e2e'], cwd=repo)
subprocess.run(['git', 'push'], cwd=repo)
```

### Agregar integrante local
```python
'nombre_usuario': Path(r'ruta\al\proyecto'),
```
